# Apply Random Forest Model to All NHDAs

This notebook:
1. Extracts full NDVI features (median, std, iqr, local_var, morans_i) for all NHDAs
2. Loads the trained Random Forest model
3. Applies the model to detect construction start years
4. Flags left-censored cases where the detected start year coincides with the first available Sentinel-2 image for that NHDA (see Methodology below)
5. Exports results with confidence scores

Author: Agnes Zwick
Date: March 2026

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

print('='*70)
print('RANDOM FOREST MODEL APPLICATION TO ALL NHDAs')
print('='*70)

## Configuration

In [ ]:
# Paths
INPUT_GPKG = r'C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas\NHDA_residential_wsf2015_max10pct.gpkg'
NDVI_DIR = r'C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\DatasetSpecific\Sentinel_2_WASP_v2\NDVI_Bavaria\Bayern_Final\masked'
MODEL_DIR = r'C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Construction_Time_Estimate'
OUTPUT_DIR = MODEL_DIR

# Model files (saved by training notebook)
RF_MODEL_PATH = Path(MODEL_DIR) / 'rf_model_construction_start.pkl'
THRESHOLDS_CSV = Path(MODEL_DIR) / 'training_ndvi_thresholds.csv'

# ID column name
ID_COL = 'nhda_id'  # unique NHDA identifier

print(f'Input GeoPackage: {INPUT_GPKG}')
print(f'NDVI Directory: {NDVI_DIR}')
print(f'Model Directory: {MODEL_DIR}')
print(f'Output Directory: {OUTPUT_DIR}')

## Load Data

In [ ]:
print('\nLoading data...')

# Load GeoPackage
gdf = gpd.read_file(INPUT_GPKG)
print(f'  Loaded {len(gdf)} NHDAs from GeoPackage')
print(f'  CRS: {gdf.crs}')

# Find NDVI rasters
ndvi_files = sorted(Path(NDVI_DIR).glob('Bayern_NDVI_*median_25832.tif'))
ndvi_map = {}
for f in ndvi_files:
    year = int(f.stem.split('_')[2])
    ndvi_map[year] = [f]
print(f'  Found NDVI rasters for years: {sorted(ndvi_map.keys())}')

# Load Random Forest model
if RF_MODEL_PATH.exists():
    with open(RF_MODEL_PATH, 'rb') as f:
        rf_model = pickle.load(f)
    print(f'  \u2713 Loaded Random Forest model from {RF_MODEL_PATH.name}')
else:
    raise FileNotFoundError(f'Random Forest model not found at {RF_MODEL_PATH}')

# Load thresholds
if THRESHOLDS_CSV.exists():
    thresholds = pd.read_csv(THRESHOLDS_CSV, index_col='threshold')['value'].to_dict()
    print(f'  \u2713 Loaded thresholds from {THRESHOLDS_CSV.name}')
else:
    print(f'  \u26a0 Thresholds file not found, will use model predictions only')
    thresholds = {}

## NDVI Feature Extraction Functions

In [ ]:
def extract_ndvi_stats(geom, raster_paths, geom_crs):
    """
    Extract NDVI statistics for a polygon from raster(s).
    Returns dict with: median, std, iqr, local_var, morans_i
    """
    for raster_path in raster_paths:
        try:
            with rasterio.open(raster_path) as src:
                # Reproject if needed
                if geom_crs != src.crs:
                    from shapely.ops import transform
                    import pyproj
                    project = pyproj.Transformer.from_crs(
                        geom_crs, src.crs, always_xy=True
                    ).transform
                    geom_reproj = transform(project, geom)
                else:
                    geom_reproj = geom

                # Check intersection
                from shapely.geometry import box
                raster_bounds = box(*src.bounds)
                if not geom_reproj.intersects(raster_bounds):
                    continue

                # Extract
                nodata = src.nodata
                fill = nodata if nodata is not None else -9999
                out_image, _ = rio_mask(
                    src, [geom_reproj], crop=True, nodata=fill, all_touched=True
                )

                arr = out_image[0].astype(np.float32)

                # Build valid-pixel mask (NDVI scaled -100 to 100)
                valid_mask = (arr >= -100.0) & (arr <= 100.0)
                if nodata is not None:
                    if np.isnan(nodata):
                        valid_mask &= ~np.isnan(arr)
                    else:
                        valid_mask &= (arr != nodata)
                arr = np.where(valid_mask, arr, np.nan)
                valid = arr[~np.isnan(arr)]

                # Scale back to -1 to 1 range
                valid = valid / 100.0

                if valid.size < 10:
                    return None

                # Moran's I (vectorised, 4-neighbour)
                def morans_i(a):
                    x = a.copy()
                    mask = ~np.isnan(x)
                    if mask.sum() < 10:
                        return np.nan
                    mean = np.nanmean(x)
                    dev = x - mean
                    right = np.roll(dev, -1, axis=1)
                    down = np.roll(dev, -1, axis=0)
                    mask_r = mask & np.roll(mask, -1, axis=1)
                    mask_d = mask & np.roll(mask, -1, axis=0)
                    num = (
                        np.nansum(dev[mask_r] * right[mask_r]) +
                        np.nansum(dev[mask_d] * down[mask_d])
                    )
                    w = mask_r.sum() + mask_d.sum()
                    den = np.nansum(dev[mask] ** 2)
                    if den == 0 or w == 0:
                        return np.nan
                    return float((mask.sum() / w) * (num / den))

                # Local variance (neighbor differences)
                def neighbor_diff_sq(a):
                    mask = ~np.isnan(a)
                    if mask.sum() < 10:
                        return np.nan
                    right = np.roll(a, -1, axis=1)
                    down = np.roll(a, -1, axis=0)
                    mask_r = mask & np.roll(mask, -1, axis=1)
                    mask_d = mask & np.roll(mask, -1, axis=0)
                    diff_r = (a - right) ** 2
                    diff_d = (a - down) ** 2
                    return float(
                        (np.nansum(diff_r[mask_r]) + np.nansum(diff_d[mask_d])) /
                        (mask_r.sum() + mask_d.sum())
                    )

                return {
                    'median': float(np.median(valid)),
                    'std': float(np.std(valid)),
                    'iqr': float(np.percentile(valid, 75) - np.percentile(valid, 25)),
                    'local_var': neighbor_diff_sq(arr),
                    'morans_i': morans_i(arr),
                }

        except Exception:
            continue

    return None


def extract_all_time_series(gdf, ndvi_map):
    """Extract NDVI time series for all NHDAs."""
    years = sorted(ndvi_map.keys())
    crs = gdf.crs
    records = []
    n = len(gdf)

    print(f'\nExtracting NDVI features for {n} NHDAs x {len(years)} years...')
    for i, (_, row) in enumerate(gdf.iterrows()):
        if i % 50 == 0:
            print(f'  [{i+1}/{n}]')

        for year in years:
            stats = extract_ndvi_stats(row.geometry, ndvi_map[year], crs)
            records.append({
                'nhda_id': row[ID_COL],
                'year': year,
                'median': stats['median'] if stats else np.nan,
                'std': stats['std'] if stats else np.nan,
                'iqr': stats['iqr'] if stats else np.nan,
                'local_var': stats['local_var'] if stats else np.nan,
                'morans_i': stats['morans_i'] if stats else np.nan,
            })

    df = pd.DataFrame(records)
    print(f'  Extracted {len(df)} records')
    print(f'  Valid records: {(~df["median"].isna()).sum()}')
    return df

print('\u2713 Feature extraction functions defined')

## Extract Time Series for All NHDAs

In [ ]:
start_time = datetime.now()

# Extract time series
ts_df = extract_all_time_series(gdf, ndvi_map)

# Save raw time series
ts_output = Path(OUTPUT_DIR) / 'all_nhdas_ndvi_timeseries.csv'
ts_df.to_csv(ts_output, index=False)
print(f'\n\u2713 Saved time series to {ts_output.name}')

elapsed = datetime.now() - start_time
print(f'  Time elapsed: {elapsed}')

## Prepare Features for Random Forest

In [ ]:
print('\nPreparing features for Random Forest...')

# Compute year-to-year changes
ts_df = ts_df.sort_values(['nhda_id', 'year']).reset_index(drop=True)

# Compute NDVI change (year-to-year difference)
ts_df['ndvi_change'] = ts_df.groupby('nhda_id')['median'].diff().fillna(0)

ts_df['ndvi_rebound'] = ts_df.groupby('nhda_id')['median'].transform(
    lambda x: x.shift(-1) - x
)

def calc_sustained(group, window=3):
    vals = group.values
    result = []
    for i in range(len(vals)):
        future = vals[i+1 : i+1+window]
        future_valid = future[~np.isnan(future.astype(float))]
        result.append(float(np.mean(future_valid)) if len(future_valid) > 0 else np.nan)
    return pd.Series(result, index=group.index)

ts_df['ndvi_sustained_low'] = ts_df.groupby('nhda_id')['median'].transform(calc_sustained)


# Rename columns to match RF model training
feature_cols = ['median', 'std', 'morans_i', 'ndvi_change']
ts_df_renamed = ts_df.rename(columns={
    'median': 'median_ndvi',
    'std': 'std_ndvi',
    'morans_i': 'morans_i'
})

# Filter valid data
rf_features = ['median_ndvi', 'std_ndvi', 'morans_i', 'ndvi_change', 'ndvi_rebound', 'ndvi_sustained_low']
ts_valid = ts_df_renamed.dropna(subset=rf_features).copy()

print(f'  Total records: {len(ts_df_renamed)}')
print(f'  Valid for RF prediction: {len(ts_valid)}')
print(f'  Features: {rf_features}')

## Apply Random Forest Model

In [ ]:
print('\nApplying Random Forest model...')

# Predict
X = ts_valid[rf_features].values
predictions = rf_model.predict(X)
probabilities = rf_model.predict_proba(X)

# Add predictions to dataframe
ts_valid['rf_prediction'] = predictions  # 0 = pre/post, 1 = under_construction
ts_valid['rf_prob_construction'] = probabilities[:, 1]  # Probability of construction

print(f'  Predictions completed for {len(ts_valid)} records')
print(f'  Predicted as construction: {(predictions == 1).sum()} ({(predictions == 1).sum()/len(predictions)*100:.1f}%)')

## Methodology: Left-Censoring and the "Already Under Construction" (AUC) Flag

Sentinel-2-derived NDVI coverage does not start in the same year for every NHDA. Depending on tile
availability and cloud masking, some polygons have their first valid NDVI observation in **2015**,
while others only become observable from **2016** onward.

The Random Forest model flags the **first year** in which a polygon's NDVI signature looks like
"under construction". This works well when that first flagged year lies safely inside the
observation period. But if the first flagged construction year is *also* the very first year we
have any valid NDVI data for that NHDA, we have a **left-censoring** problem: we cannot tell
whether construction genuinely began that year, or whether it had already started earlier and we
simply have no imagery to see it.

**Rule applied below:**
- For every NHDA, determine `first_available_year` – the earliest year with a valid (non-missing)
  NDVI extraction, independent of the RF prediction.
- If the RF-detected start year equals `first_available_year`, the case is marked
  `left_censored = True` and the **`construction_start_year` field itself is written as the label**
  `AUC_<year>` ("Already Under Construction" at the first available image) instead of a plain number:
  - Detected start = 2015 and first image = 2015 → `construction_start_year = 'AUC_2015'`
  - Detected start = 2016 and first image = 2016 → `construction_start_year = 'AUC_2016'`
- If the detected start year is later than `first_available_year`, there is at least one earlier
  "not yet under construction" observation to anchor the estimate, so `construction_start_year`
  holds the plain year (as text, e.g. `'2018'`) and `left_censored = False`.

## Detect Construction Start Year per NHDA

In [ ]:
print('\nDetecting construction start year per NHDA...')

# First year with any valid NDVI observation per NHDA (independent of RF prediction)
# -> used below to flag left-censored / "already under construction" (AUC) cases
first_available = (
    ts_df.dropna(subset=['median'])
    .groupby('nhda_id')['year']
    .min()
)
print(f'  First-available-image year computed for {len(first_available)} NHDAs')
print(f'  Distribution of first available year:')
for year, count in first_available.value_counts().sort_index().items():
    print(f'    {int(year)}: {count}')


def detect_construction_year(nda_df, first_avail_year):
    """
    For each NHDA, find the first year predicted as 'under_construction'.

    Also checks for left-censoring: if that first detected year is the same as
    the first year we have any valid NDVI observation for this NHDA, we cannot
    rule out that construction started even earlier, before our Sentinel-2
    record begins. In that case 'construction_start_year' is written as the
    label 'AUC_<year>' ("Already Under Construction" at the first available
    image) instead of a plain year.

    Returns: construction_start_year (plain year or 'AUC_<year>'), confidence
    (based on probability), left-censoring flag
    """
    construction_years = nda_df[nda_df['rf_prediction'] == 1].copy()

    if len(construction_years) == 0:
        return pd.Series({
            'construction_start_year': None,
            'first_available_year': first_avail_year,
            'left_censored': False,
            'rf_confidence': None,
            'method': 'no_detection',
            'n_construction_years': 0
        })

    # First year predicted as construction
    first_construction = construction_years.iloc[0]
    detected_year = int(first_construction['year'])

    # Confidence based on probability
    prob = first_construction['rf_prob_construction']
    if prob >= 0.8:
        confidence = 'high'
    elif prob >= 0.6:
        confidence = 'medium'
    else:
        confidence = 'low'

    # Left-censoring check: detected start == first available image year means
    # construction may already have been underway before our record starts.
    # In that case, construction_start_year itself becomes the AUC label.
    if first_avail_year is not None and detected_year == int(first_avail_year):
        left_censored = True
        construction_start_year = f'AUC_{detected_year}'  # Already Under Construction
    else:
        left_censored = False
        construction_start_year = str(detected_year)

    return pd.Series({
        'construction_start_year': construction_start_year,
        'first_available_year': first_avail_year,
        'left_censored': left_censored,
        'rf_confidence': confidence,
        'rf_probability': float(prob),
        'method': 'random_forest',
        'n_construction_years': len(construction_years)
    })

# Apply to each NHDA - use list comprehension for better compatibility
result_list = []
for nhda_id, group in ts_valid.groupby('nhda_id'):
    fav = first_available.get(nhda_id, None)
    result = detect_construction_year(group, fav)
    result['nhda_id'] = nhda_id
    result_list.append(result)

results = pd.DataFrame(result_list)

print(f'  Processed {len(results)} NHDAs')
print(f'  Construction detected: {(~results["construction_start_year"].isna()).sum()}')
print(f'  No detection: {(results["construction_start_year"].isna()).sum()}')
print(f'  Left-censored (already under construction at first image): {results["left_censored"].sum()}')

# Distribution by construction_start_year (plain years and AUC_<year> labels together)
if (~results['construction_start_year'].isna()).sum() > 0:
    print('\n  construction_start_year distribution:')
    year_dist = results['construction_start_year'].value_counts().sort_index()
    for label, count in year_dist.items():
        print(f'    {label}: {count}')

# Confidence distribution
print('\n  Confidence distribution:')
conf_dist = results['rf_confidence'].value_counts()
for conf, count in conf_dist.items():
    print(f'    {conf}: {count}')

## Merge with Original GeoPackage and Export

In [ ]:
print('\nMerging results with original GeoPackage...')

# Merge with original geometries
gdf_result = gdf.merge(results, left_on=ID_COL, right_on='nhda_id', how='left')

# Fill NaN for NHDAs without detection
gdf_result['method'] = gdf_result['method'].fillna('no_data')
gdf_result['construction_start_year'] = gdf_result['construction_start_year'].fillna('no_data')
gdf_result['left_censored'] = gdf_result['left_censored'].fillna(False)

# Export to GeoPackage
output_gpkg = Path(OUTPUT_DIR) / 'NHDA_with_construction_years_RF_v2.gpkg'
gdf_result.to_file(output_gpkg, driver='GPKG')
print(f'  \u2713 Saved GeoPackage: {output_gpkg.name}')

# Export to CSV
output_csv = Path(OUTPUT_DIR) / 'NHDA_construction_years_RF_v2.csv'
results_csv = gdf_result[[ID_COL, 'construction_start_year', 'first_available_year',
                          'left_censored', 'rf_confidence',
                          'rf_probability', 'method', 'n_construction_years']].copy()
results_csv.to_csv(output_csv, index=False)
print(f'  \u2713 Saved CSV: {output_csv.name}')

print('\n' + '='*70)
print('DONE!')
print('='*70)
print(f'Total time: {datetime.now() - start_time}')
print(f'\nOutputs saved to: {OUTPUT_DIR}')

## Summary Statistics

In [ ]:
print('\nSUMMARY STATISTICS')
print('='*70)

total = len(gdf_result)
detected = (gdf_result['construction_start_year'] != 'no_data').sum()
no_detection = (gdf_result['construction_start_year'] == 'no_data').sum()
left_censored = gdf_result['left_censored'].sum()

print(f'Total NHDAs: {total}')
print(f'Construction detected: {detected} ({detected/total*100:.1f}%)')
print(f'No detection: {no_detection} ({no_detection/total*100:.1f}%)')
print(f'  of which left-censored / AUC (already under construction at first image): '
      f'{left_censored} ({left_censored/total*100:.1f}%)')

print('\nDetection method breakdown:')
method_counts = gdf_result['method'].value_counts()
for method, count in method_counts.items():
    print(f'  {method}: {count} ({count/total*100:.1f}%)')

if detected > 0:
    print('\nConfidence distribution (for detected):')
    conf_counts = gdf_result[gdf_result['construction_start_year'] != 'no_data']['rf_confidence'].value_counts()
    for conf, count in conf_counts.items():
        print(f'  {conf}: {count} ({count/detected*100:.1f}%)')

    print('\nconstruction_start_year distribution (plain years and AUC_<year> labels):')
    year_counts = gdf_result.loc[gdf_result['construction_start_year'] != 'no_data',
                                  'construction_start_year'].value_counts().sort_index()
    for label, count in year_counts.items():
        print(f'  {label}: {count}')